# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Godność osoby ludzkiej
2. Pan Wołodyjowski
3. Nie budźcie mnie
4. Szymborska. Kropki, przecinki, papierosy
5. Wspólnie. 20 lat kolektywu Sputnik Photos
6. Rodzeństwo (Ritter, Dene, Voss)
7. Męczeństwo i śmierć Marata
8. Gotyk w Karpatach
9. Fisz Emade Tworzywo w Studio
10. Nie ma jak w domu. Projekt fotograficzny Agaty Mendziuk

Czas wykonania: 3.66s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Godność osoby ludzkiej
2. Pan Wołodyjowski
3. Nie budźcie mnie
4. Szymborska. Kropki, przecinki, papierosy
5. Wspólnie. 20 lat kolektywu Sputnik Photos
6. Rodzeństwo (Ritter, Dene, Voss)
7. Męczeństwo i śmierć Marata
8. Gotyk w Karpatach
9. Fisz Emade Tworzywo w Studio
10. Nie ma jak w domu. Projekt fotograficzny Agaty Mendziuk

Czas wykonania (wątki): 1.04s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [4]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [5]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 12 procesach (rdzeniach)...
Zakończono w czasie 0.76s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [6]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_FACT_API_URL = "https://catfact.ninja/fact"


def fetch_cat_fact():
    response = requests.get(CAT_FACT_API_URL)
    return response.json().get("fact")


def fetch_sequential(count=20):
    facts_list = []
    start_time = time.time()

    for _ in range(count):
        facts_list.append(fetch_cat_fact())

    elapsed_time = time.time() - start_time
    return facts_list, elapsed_time

def fetch_threaded(count=20, max_workers=10):
    start_time = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = executor.map(lambda _: fetch_cat_fact(), range(count))
        facts_list = list(results)
    elapsed_time = time.time() - start_time
    return facts_list, elapsed_time


def main():
    print("Sequential")
    _, seq_time = fetch_sequential()
    print(f"Sequential time: {seq_time:.3f} s\n")

    print("Multithread")
    _, thr_time = fetch_threaded()
    print(f"Multithread time: {thr_time:.3f} s\n")

    print("Comparation")
    print(f"{seq_time / thr_time:.2f} x faster")


if __name__ == "__main__":
    main()

Sequential
Sequential time: 6.929 s

Multithread
Multithread time: 1.693 s

Comparation
4.09 x faster


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [8]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

# Twój kod tutaj...

TOTAL_NUMBERS = 30
q = queue.Queue()

def producer():
    for i in range(1, TOTAL_NUMBERS + 1):
        q.put(i)
        print(f"producer add {i}")
        time.sleep(0.05)

    q.put(None)
    q.put(None)


def consumer_even():
    while True:
        item = q.get()
        if item is None:
            break
        if item % 2 == 0:
            print(f"consumer even received: {item}")
        q.task_done()

def consumer_odd():
    while True: 
        item = q.get()
        if item is None:
            break
        if item % 2 == 1:
            print(f" consumer odd received {item}")
        q.task_done()

def main():
    
    t_prod = threading.Thread(target=producer)
    t_even = threading.Thread(target=consumer_even)
    t_odd = threading.Thread(target=consumer_odd)

    t_prod.start()
    t_even.start()
    t_odd.start()
    t_prod.join()
    t_even.join()
    t_odd.join()
    print("End")

if __name__ == "__main__":
    main()



producer add 1
producer add 2
consumer even received: 2
producer add 3
 consumer odd received 3
producer add 4
consumer even received: 4
producer add 5
 consumer odd received 5
producer add 6consumer even received: 6

producer add 7
 consumer odd received 7
producer add 8
consumer even received: 8
producer add 9
 consumer odd received 9
producer add 10
consumer even received: 10
producer add 11 consumer odd received 11

producer add 12
consumer even received: 12
producer add 13
 consumer odd received 13
producer add 14
consumer even received: 14
producer add 15
 consumer odd received 15
producer add 16consumer even received: 16

producer add 17
 consumer odd received 17
producer add 18
consumer even received: 18
producer add 19
 consumer odd received 19
producer add 20
consumer even received: 20
producer add 21 consumer odd received 21

producer add 22
consumer even received: 22
producer add 23
 consumer odd received 23
producer add 24
consumer even received: 24
producer add 25
 consum

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [9]:
# Space for task 3 solution
import multiprocessing
import time
from lab2_functions import calculate_power_sum

START_RANGE = 1
END_RANGE = 10_000


def run_sequential():
    start_time = time.time()

    results = []
    for value in range(START_RANGE, END_RANGE + 1):
        results.append(calculate_power_sum(value))

    elapsed_time = time.time() - start_time
    return results, elapsed_time


def run_parallel():
    start_time = time.time()

    with multiprocessing.Pool() as pool:
        results = pool.map(
            calculate_power_sum,
            range(START_RANGE, END_RANGE + 1)
        )

    elapsed_time = time.time() - start_time
    return results, elapsed_time


if __name__ == "__main__":
    print("Sequential computation")
    _, seq_time = run_sequential()
    print(f"Sequential time: {seq_time:.3f} s\n")

    print("Parallel computation (multiprocessing)")
    _, par_time = run_parallel()
    print(f"Parallel time: {par_time:.3f} s\n")

    print("Comparison")
    print(f"Speedup: {seq_time / par_time:.2f} x faster ")

    pass

Sequential computation
Sequential time: 0.375 s

Parallel computation (multiprocessing)
Parallel time: 0.196 s

Comparison
Speedup: 1.91 x faster 
